# BigQuery Anti-Pattern Recognition - Complete Deployment Guide

This notebook provides a comprehensive guide to deploy the BigQuery Anti-Pattern Recognition tool using **three different approaches**:

## 🎯 Deployment Scenarios

### 1. **Cloud Run Job (Batch Processing)**
- **Use Case**: Scheduled analysis of query history from INFORMATION_SCHEMA
- **Best For**: Regular audits, automated monitoring, processing historical data
- **Output**: Results written to BigQuery table

### 2. **Cloud Run Service (REST API)**
- **Use Case**: Real-time anti-pattern detection via HTTP API
- **Best For**: CI/CD integration, interactive tools, on-demand analysis
- **Output**: JSON response with anti-patterns and recommendations

### 3. **BigQuery Remote UDF**
- **Use Case**: SQL-native anti-pattern detection within BigQuery
- **Best For**: Data analysts, SQL-based workflows, ad-hoc analysis
- **Output**: JSON result directly in SQL queries

---

## 📋 What This Notebook Does

1. **Setup & Configuration**: Configure your Google Cloud project settings
2. **Prerequisites Check**: Verify APIs, authentication, and permissions
3. **Container Building**: Build optimized containers for each deployment type
4. **Multi-Scenario Deployment**: Deploy all three scenarios with proper configuration
5. **Comprehensive Testing**: Test each deployment with real examples
6. **Usage Demonstrations**: Show practical examples for each scenario

---

## 🔧 Prerequisites

- Google Cloud Project with billing enabled
- `gcloud` CLI installed and authenticated
- Required APIs will be enabled automatically
- Sufficient IAM permissions for Cloud Run, BigQuery, and Artifact Registry

---

**⚡ Ready to get started? Let's deploy all three scenarios!**

## Step 1: Install Dependencies and Setup

In [ ]:
# Install required packages
!pip install -r requirements.txt -q

print("✅ Dependencies installed successfully!")

In [ ]:
# Import required libraries
import os
import sys
import json
import time
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Import our utilities
from utils import ConfigManager, DeploymentHelper, AntiPatternAnalyzer, SampleQueries

print("✅ Libraries imported successfully!")

## Step 2: Configuration Setup

Configure your Google Cloud project settings for all deployment scenarios:

In [ ]:
# Initialize configuration manager
config = ConfigManager()

# Create configuration widgets
project_id_widget = widgets.Text(
    value=config.get('project_id', ''),
    description='Project ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

region_widget = widgets.Dropdown(
    options=['us-central1', 'us-east1', 'us-west1',
             'europe-west1', 'asia-southeast1'],
    value=config.get('region', 'us-central1'),
    description='Region:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Service names for different deployments
batch_service_widget = widgets.Text(
    value=config.get('batch_service_name', 'antipattern-batch-job'),
    description='Batch Job Name:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

api_service_widget = widgets.Text(
    value=config.get('api_service_name', 'antipattern-api-service'),
    description='API Service Name:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

artifact_registry_widget = widgets.Text(
    value=config.get('artifact_registry', 'antipattern-registry'),
    description='Artifact Registry:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

bq_dataset_widget = widgets.Text(
    value=config.get('bq_dataset', 'antipattern_demo'),
    description='BQ Dataset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Deployment selection checkboxes
deploy_batch_widget = widgets.Checkbox(
    value=config.get('deploy_batch', True),
    description='Deploy Cloud Run Job (Batch Processing)',
    style={'description_width': 'initial'}
)

deploy_api_widget = widgets.Checkbox(
    value=config.get('deploy_api', True),
    description='Deploy Cloud Run Service (REST API)',
    style={'description_width': 'initial'}
)

deploy_udf_widget = widgets.Checkbox(
    value=config.get('deploy_udf', True),
    description='Deploy BigQuery Remote UDF',
    style={'description_width': 'initial'}
)

# Display configuration form
print("📝 Configure your deployment settings:")
display(widgets.VBox([
    widgets.HTML("<h3>🔧 Project Configuration</h3>"),
    project_id_widget,
    region_widget,
    artifact_registry_widget,
    bq_dataset_widget,
    widgets.HTML("<h3>🚀 Service Names</h3>"),
    batch_service_widget,
    api_service_widget,
    widgets.HTML("<h3>📦 Deployment Selection</h3>"),
    deploy_batch_widget,
    deploy_api_widget,
    deploy_udf_widget
]))

In [ ]:
# Save configuration
config.update({
    'project_id': project_id_widget.value,
    'region': region_widget.value,
    'batch_service_name': batch_service_widget.value,
    'api_service_name': api_service_widget.value,
    'artifact_registry': artifact_registry_widget.value,
    'bq_dataset': bq_dataset_widget.value,
    'deploy_batch': deploy_batch_widget.value,
    'deploy_api': deploy_api_widget.value,
    'deploy_udf': deploy_udf_widget.value,
    'deployment_timestamp': datetime.now().isoformat()
})

# Generate container image names
artifact_registry_url = f"{config.get('region')}-docker.pkg.dev/{config.get('project_id')}/{config.get('artifact_registry')}"
batch_image = f"{artifact_registry_url}/antipattern-batch:latest"
service_image = f"{artifact_registry_url}/antipattern-service:latest"

config.update({
    'artifact_registry_url': artifact_registry_url,
    'batch_container_image': batch_image,
    'service_container_image': service_image
})

print(f"✅ Configuration saved!")
print(f"Project ID: {config.get('project_id')}")
print(f"Region: {config.get('region')}")
print(f"Deployments selected:")
if config.get('deploy_batch'):
    print(f"  ✓ Batch Job: {config.get('batch_service_name')}")
if config.get('deploy_api'):
    print(f"  ✓ API Service: {config.get('api_service_name')}")
if config.get('deploy_udf'):
    print(f"  ✓ BigQuery UDF")

## Step 3: Prerequisites Check

Verify that your environment is ready for deployment:

In [ ]:
# Check gcloud authentication
print("🔐 Checking gcloud authentication...")
if DeploymentHelper.check_gcloud_auth():
    print("✅ gcloud is authenticated")
else:
    print("❌ gcloud is not authenticated")
    print("Please run: gcloud auth login")
    print("And: gcloud auth application-default login")

In [ ]:
# Check required APIs
required_apis = [
    'cloudbuild.googleapis.com',
    'run.googleapis.com',
    'artifactregistry.googleapis.com',
    'bigquery.googleapis.com',
    'bigqueryconnection.googleapis.com'
]

print("🔍 Checking required APIs...")
api_status = DeploymentHelper.check_apis_enabled(
    config.get('project_id'), required_apis)

all_enabled = True
for api, enabled in api_status.items():
    status = "✅" if enabled else "❌"
    print(f"{status} {api}: {'Enabled' if enabled else 'Disabled'}")
    if not enabled:
        all_enabled = False

if not all_enabled:
    print("\n⚠️  Some APIs are not enabled. Enable them with:")
    print(
        f"gcloud services enable {' '.join(required_apis)} --project={config.get('project_id')}")
else:
    print("\n✅ All required APIs are enabled!")

In [ ]:
# Enable APIs if needed
enable_apis_btn = widgets.Button(
    description='Enable Required APIs',
    button_style='warning',
    layout=widgets.Layout(width='200px')
)

output_widget = widgets.Output()


def on_enable_apis(b):
    with output_widget:
        clear_output()
        print("🔄 Enabling APIs...")

        command = f"gcloud services enable {' '.join(required_apis)} --project={config.get('project_id')}"
        success, output = DeploymentHelper.run_command(command)

        if success:
            print("✅ APIs enabled successfully!")
        else:
            print(f"❌ Failed to enable APIs: {output}")


enable_apis_btn.on_click(on_enable_apis)

display(widgets.VBox([enable_apis_btn, output_widget]))

## Step 4: Create Artifact Registry Repository

Create a repository to store our container images:

In [ ]:
# Create Artifact Registry repository
print("📦 Creating Artifact Registry repository...")

command = f"""
gcloud artifacts repositories create {config.get('artifact_registry')} \
    --repository-format=docker \
    --location={config.get('region')} \
    --description="BigQuery Anti-Pattern Recognition" \
    --project={config.get('project_id')}
"""

success, output = DeploymentHelper.run_command(command)

if success or "already exists" in output:
    print("✅ Artifact Registry repository ready!")
    print(f"Repository URL: {config.get('artifact_registry_url')}")
else:
    print(f"❌ Failed to create repository: {output}")

## Step 5: Build Container Images

Build optimized containers for each deployment scenario:

In [ ]:
# Build batch processing container (if selected)
if config.get('deploy_batch'):
    print("🔨 Building batch processing container...")
    print("This container uses the Main class for command-line batch processing.")
    print("⏱️  This may take 5-10 minutes...")

    batch_build_command = f"""
    cd .. && \
    gcloud builds submit . \
        --project={config.get('project_id')} \
        --config=demo/cloudbuild-batch.yaml \
        --substitutions=_CONTAINER_IMAGE_NAME={config.get('batch_container_image')} \
        --machine-type=e2-highcpu-8
    """

    success, output = DeploymentHelper.run_command(batch_build_command)

    if success:
        print("✅ Batch container built successfully!")
        print(f"Image: {config.get('batch_container_image')}")
    else:
        print(f"❌ Batch build failed: {output}")
        print("\nTroubleshooting tips:")
        print("1. Make sure you're running this from the demo/ directory")
        print("2. Check that demo/cloudbuild-batch.yaml exists")
        print("3. Verify your project has Cloud Build API enabled")
else:
    print("⏭️  Skipping batch container build (not selected)")

In [ ]:
# Build service container (if API or UDF selected)
if config.get('deploy_api') or config.get('deploy_udf'):
    print("🔨 Building service container...")
    print("This container uses the AntiPatternApplication class for web service/UDF.")
    print("⏱️  This may take 5-10 minutes...")

    service_build_command = f"""
    cd .. && \
    gcloud builds submit . \
        --project={config.get('project_id')} \
        --config=demo/cloudbuild-service.yaml \
        --substitutions=_CONTAINER_IMAGE_NAME={config.get('service_container_image')} \
        --machine-type=e2-highcpu-8
    """

    success, output = DeploymentHelper.run_command(service_build_command)

    if success:
        print("✅ Service container built successfully!")
        print(f"Image: {config.get('service_container_image')}")
    else:
        print(f"❌ Service build failed: {output}")
        print("\nTroubleshooting tips:")
        print("1. Make sure you're running this from the demo/ directory")
        print("2. Check that demo/cloudbuild-service.yaml exists")
        print("3. Verify your project has Cloud Build API enabled")
else:
    print("⏭️  Skipping service container build (not selected)")

## Step 6A: Deploy Cloud Run Job (Batch Processing)

Deploy the batch processing job for scheduled INFORMATION_SCHEMA analysis:

In [ ]:
# Step 6A: Deploy Cloud Run Job (Batch Processing) - FIXED VERSION V2
# Copy this code into your Jupyter notebook cell

if config.get('deploy_batch'):
    print("🚀 Deploying Cloud Run Job for batch processing...")

    # Create output table for batch results
    output_table = f"{config.get('project_id')}.{config.get('bq_dataset')}.antipattern_batch_results"

    # Check if job already exists
    check_job_command = f"""
    gcloud run jobs describe {config.get('batch_service_name')} \
        --region={config.get('region')} \
        --project={config.get('project_id')} \
        --format="value(metadata.name)"
    """

    job_exists, _ = DeploymentHelper.run_command(check_job_command)

    if job_exists:
        print(
            f"⚠️  Cloud Run Job '{config.get('batch_service_name')}' already exists.")
        print("Deleting existing job and creating new one...")

        # Delete existing job
        delete_command = f"""
        gcloud run jobs delete {config.get('batch_service_name')} \
            --region={config.get('region')} \
            --project={config.get('project_id')} \
            --quiet
        """

        delete_success, delete_output = DeploymentHelper.run_command(
            delete_command)
        if delete_success:
            print("✅ Existing job deleted successfully!")
        else:
            print(f"⚠️  Could not delete existing job: {delete_output}")

    # FIXED: Use proper INFORMATION_SCHEMA table name without backticks
    # The backticks are causing issues when passed through command line arguments
    project_id = config.get('project_id')

    # Use the format that works with the BigQuery client
    info_schema_table = f"{project_id}.region-us.INFORMATION_SCHEMA.JOBS"

    batch_deploy_command = f"""
    gcloud run jobs create {config.get('batch_service_name')} \
        --image={config.get('batch_container_image')} \
        --max-retries=3 \
        --task-timeout=15m \
        --memory=2Gi \
        --cpu=2 \
        --args="--read_from_info_schema" \
        --args="--read_from_info_schema_days" --args="1" \
        --args="--info_schema_table_name" --args="{info_schema_table}" \
        --args="--processing_project_id" --args="{project_id}" \
        --args="--output_table" --args="{output_table}" \
        --region={config.get('region')} \
        --project={project_id}
    """

    success, output = DeploymentHelper.run_command(batch_deploy_command)

    if success:
        print("✅ Cloud Run Job deployed successfully!")
        config.set('batch_job_name', config.get('batch_service_name'))
        config.set('batch_output_table', output_table)
        print(f"Job Name: {config.get('batch_service_name')}")
        print(f"Output Table: {output_table}")
        print(f"INFORMATION_SCHEMA Table: {info_schema_table}")
    else:
        print(f"❌ Batch job deployment failed: {output}")
        print("\nTroubleshooting tips:")
        print("1. Check that the container image was built successfully")
        print("2. Verify Cloud Run API is enabled")
        print("3. Ensure sufficient IAM permissions")
        print("4. Check the Cloud Run logs in Google Cloud Console")

else:
    print("⏭️  Skipping Cloud Run Job deployment (not selected)")

## Step 6B: Deploy Cloud Run Service (REST API)

Deploy the REST API service for real-time anti-pattern detection:

In [ ]:
if config.get('deploy_api'):
    print("🚀 Deploying Cloud Run Service for REST API...")

    api_deploy_command = f"""
    gcloud run deploy {config.get('api_service_name')} \
        --image={config.get('service_container_image')} \
        --region={config.get('region')} \
        --no-allow-unauthenticated \
        --memory=2Gi \
        --cpu=2 \
        --timeout=300 \
        --port=8080 \
        --project={config.get('project_id')}
    """

    success, output = DeploymentHelper.run_command(api_deploy_command)

    if success:
        print("✅ Cloud Run Service deployed successfully!")

        # Get service URL
        url_command = f"""
        gcloud run services describe {config.get('api_service_name')} \
            --region={config.get('region')} \
            --project={config.get('project_id')} \
            --format="value(status.address.url)"
        """

        success, service_url = DeploymentHelper.run_command(url_command)
        if success:
            config.set('api_service_url', service_url.strip())
            print(f"Service URL: {config.get('api_service_url')}")
    else:
        print(f"❌ API service deployment failed: {output}")
else:
    print("⏭️  Skipping Cloud Run Service deployment (not selected)")

## Step 7: Setup BigQuery Dataset and Components

Create BigQuery dataset and prepare for UDF deployment:

In [ ]:
# Create BigQuery dataset
print("📊 Creating BigQuery dataset...")

dataset_command = f"""
bq mk --dataset \
    --project_id={config.get('project_id')} \
    --location={config.get('region')} \
    --description="Anti-Pattern Recognition Demo" \
    {config.get('bq_dataset')}
"""

success, output = DeploymentHelper.run_command(dataset_command)

if success or "already exists" in output:
    print("✅ BigQuery dataset ready!")
    print(f"Dataset: {config.get('project_id')}.{config.get('bq_dataset')}")
else:
    print(f"❌ Failed to create dataset: {output}")

In [ ]:
# Create BigQuery connection for UDF (if UDF deployment selected)
if config.get('deploy_udf') and config.get('api_service_url'):
    print("🔗 Creating BigQuery connection for UDF...")

    connection_name = f"ext-{config.get('api_service_name')}"

    connection_command = f"""
    bq mk --connection \
        --display_name='Anti-Pattern Recognition Connection' \
        --connection_type=CLOUD_RESOURCE \
        --project_id={config.get('project_id')} \
        --location={config.get('region')} \
        {connection_name}
    """

    success, output = DeploymentHelper.run_command(connection_command)

    if success or "already exists" in output:
        print("✅ BigQuery connection created!")
        config.set('bq_connection', connection_name)

        # Get connection service account
        sa_command = f"""
        bq --project_id={config.get('project_id')} --format=json show \
            --connection {config.get('project_id')}.{config.get('region')}.{connection_name}
        """

        success, sa_output = DeploymentHelper.run_command(sa_command)
        if success:
            import json
            connection_info = json.loads(sa_output)
            service_account = connection_info['cloudResource']['serviceAccountId']
            config.set('connection_service_account', service_account)
            print(f"Connection service account: {service_account}")
    else:
        print(f"❌ Failed to create connection: {output}")
elif config.get('deploy_udf'):
    print("⚠️  Cannot create UDF connection - API service not deployed")
else:
    print("⏭️  Skipping BigQuery connection (UDF not selected)")

In [ ]:
# Grant Cloud Run Invoker role to connection service account
if config.get('deploy_udf') and config.get('connection_service_account'):
    print("🔐 Granting permissions...")

    permission_command = f"""
    gcloud projects add-iam-policy-binding {config.get('project_id')} \
        --member="serviceAccount:{config.get('connection_service_account')}" \
        --role='roles/run.invoker'
    """

    success, output = DeploymentHelper.run_command(permission_command)

    if success:
        print("✅ Permissions granted successfully!")
    else:
        print(f"❌ Failed to grant permissions: {output}")
else:
    print("⏭️  Skipping permission grant (UDF not selected or service account not found)")

## Step 8: Create BigQuery Remote Function (UDF)

Create the UDF that will call our Cloud Run service:

In [ ]:
# Create remote function
if config.get('deploy_udf') and config.get('api_service_url'):
    print("🔧 Creating BigQuery remote function...")

    # Build the SQL statement
    dataset = config.get('bq_dataset')
    project_id = config.get('project_id')
    region = config.get('region')
    connection = config.get('bq_connection')
    service_url = config.get('api_service_url').rstrip(
        '/')  # Remove trailing slash

    # Create the SQL statement
    function_sql = f"""CREATE OR REPLACE FUNCTION {dataset}.get_antipatterns(query STRING)
RETURNS JSON
REMOTE WITH CONNECTION `{project_id}.{region}.{connection}`
OPTIONS (endpoint = '{service_url}');"""

    # Execute using file input to avoid shell interpretation issues
    import os
    import subprocess
    import tempfile

    with tempfile.NamedTemporaryFile(mode='w', suffix='.sql', delete=False) as f:
        f.write(function_sql)
        temp_sql_file = f.name

    try:
        with open(temp_sql_file, 'r') as sql_file:
            result = subprocess.run(
                ['bq', 'query',
                    f'--project_id={project_id}', '--use_legacy_sql=false'],
                stdin=sql_file,
                capture_output=True,
                text=True,
                timeout=60
            )

        if result.returncode == 0:
            print("✅ Remote function created successfully!")
            config.set('udf_name', f"{dataset}.get_antipatterns")
            print(f"UDF Name: {config.get('udf_name')}")
            print()
            print("📋 Example usage:")
            print(
                f"   SELECT {dataset}.get_antipatterns('SELECT * FROM table') as antipatterns;")
        else:
            print(f"❌ Failed to create function: {result.stderr}")
            print("Manual creation SQL:")
            print(function_sql)

    except Exception as e:
        print(f"❌ Error: {e}")
        print("Manual creation SQL:")
        print(function_sql)

    finally:
        if os.path.exists(temp_sql_file):
            os.unlink(temp_sql_file)

else:
    print("⏭️  Skipping UDF creation (not selected or API service not available)")

## Step 9: Test All Deployments

Test each deployment scenario to ensure everything is working:

In [ ]:
# Step 9: Test Cloud Run Job (Batch Processing) - FIXED VERSION
# Copy this code into your Jupyter notebook cell

# Test Cloud Run Job (Batch Processing)
if config.get('deploy_batch') and config.get('batch_job_name'):
    print("🧪 Testing Cloud Run Job (Batch Processing)...")

    # Execute the job
    execute_command = f"""
    gcloud run jobs execute {config.get('batch_job_name')} \
        --region={config.get('region')} \
        --project={config.get('project_id')} \
        --wait
    """

    print("⏱️  Executing batch job... This may take a few minutes.")
    success, output = DeploymentHelper.run_command(execute_command)

    if success:
        print("✅ Batch job executed successfully!")
        print(
            f"Results should be available in: {config.get('batch_output_table')}")

        # Check if results were written - FIXED: Remove backticks to avoid escape sequence warning
        batch_output_table = config.get('batch_output_table')
        check_results_command = f"""
        bq query --project_id={config.get('project_id')} --use_legacy_sql=false \
            "SELECT COUNT(*) as result_count FROM {batch_output_table}"
        """

        success, result_output = DeploymentHelper.run_command(
            check_results_command)
        if success:
            print(f"📊 Results check: {result_output}")
        else:
            print(f"⚠️  Could not check results: {result_output}")
    else:
        print(f"❌ Batch job execution failed: {output}")
        print("\nTroubleshooting tips:")
        print("1. Check the Cloud Run logs in Google Cloud Console")
        print("2. Verify the INFORMATION_SCHEMA table name is correct")
        print("3. Ensure the service account has BigQuery permissions")
        print("4. Check if there are queries in the last 24 hours to analyze")

else:
    print("⏭️  Skipping batch job test (not deployed)")

In [ ]:
# Step 9: Test Cloud Run Service (REST API) - Simplified Working Version
# Copy this code into your Jupyter notebook cell

# Test Cloud Run Service (REST API)
if config.get('deploy_api'):
    print("🧪 Testing Cloud Run Service (REST API)...")

    # Get service URL
    service_url = config.get('api_service_url')
    if not service_url:
        print("⚠️  Service URL not found in config. Retrieving from gcloud...")
        url_command = f"""
        gcloud run services describe {config.get('api_service_name')} \
            --region={config.get('region')} \
            --project={config.get('project_id')} \
            --format="value(status.address.url)"
        """
        success, service_url_output = DeploymentHelper.run_command(url_command)
        if success and service_url_output.strip():
            service_url = service_url_output.strip()
            config.set('api_service_url', service_url)
            print(f"✅ Retrieved service URL: {service_url}")
        else:
            print("❌ Could not retrieve service URL. Service may not be deployed.")
            service_url = None

    if service_url:
        print(f"Using service URL: {service_url}")

        import json
        import subprocess

        import requests

        # Make service publicly accessible for testing
        print("🔧 Making service publicly accessible...")
        public_command = f"""
        gcloud run services add-iam-policy-binding {config.get('api_service_name')} \
            --member="allUsers" \
            --role="roles/run.invoker" \
            --region={config.get('region')} \
            --project={config.get('project_id')}
        """

        public_result = subprocess.run(
            public_command,
            shell=True,
            capture_output=True,
            text=True
        )

        if public_result.returncode == 0:
            print("✅ Service made publicly accessible")
        else:
            print(f"⚠️  Permission command result: {public_result.stderr}")

        # Test the API
        test_query = "SELECT * FROM `bigquery-public-data.samples.shakespeare` ORDER BY word_count DESC LIMIT 10"
        print(f"Testing with query: {test_query[:50]}...")

        try:
            response = requests.post(
                service_url,
                json={'calls': [[test_query]]},
                timeout=30
            )

            print(f"Response status: {response.status_code}")

            if response.status_code == 200:
                result = response.json()
                print("✅ REST API is working!")
                print(f"Response: {str(result)[:300]}...")

                # Parse anti-patterns from response
                if 'replies' in result and result['replies']:
                    first_reply = result['replies'][0]
                    if 'antipatterns' in first_reply:
                        antipatterns = first_reply['antipatterns']
                        print(f"Found {len(antipatterns)} anti-patterns:")
                        for ap in antipatterns[:3]:
                            name = ap.get('name', 'Unknown')
                            description = ap.get('result', ap.get(
                                'description', 'No description'))
                            print(f"  - {name}: {description[:100]}...")
                    else:
                        print(f"Response format: {first_reply}")
            else:
                print(f"❌ API test failed. Status: {response.status_code}")
                print(f"Response: {response.text[:300]}...")

        except Exception as e:
            print(f"❌ API request failed: {e}")

    else:
        print("❌ No service URL available. Please deploy the API service first.")
else:
    print("⏭️  Skipping API test (not deployed)")

In [ ]:
# Test BigQuery UDF
if config.get('deploy_udf') and config.get('udf_name'):
    print("🧪 Testing BigQuery UDF...")

    try:
        test_udf_sql = f"""
        SELECT
            'SELECT * FROM dataset.table ORDER BY column' as test_query,
            {config.get('udf_name')}('SELECT * FROM dataset.table ORDER BY column') as antipatterns
        """

        # First try a dry run
        dry_run_command = f"""
        bq query --project_id={config.get('project_id')} --use_legacy_sql=false \
            --dry_run "{test_udf_sql}"
        """

        success, output = DeploymentHelper.run_command(dry_run_command)

        if success:
            print("✅ BigQuery UDF is accessible!")

            # Now try actual execution
            actual_command = f"""
            bq query --project_id={config.get('project_id')} --use_legacy_sql=false \
                "{test_udf_sql}"
            """

            success, result = DeploymentHelper.run_command(actual_command)
            if success:
                print("✅ UDF execution successful!")
                print(f"📊 UDF result preview: {result}")
            else:
                print(f"⚠️  UDF dry run passed but execution failed: {result}")
        else:
            print(f"❌ UDF test failed: {output}")

    except Exception as e:
        print(f"❌ UDF test failed: {e}")
else:
    print("⏭️  Skipping UDF test (not deployed)")

## Step 10: Demonstration Examples

Show practical examples for each deployment scenario:

In [ ]:
# Demonstrate sample queries with different anti-patterns
print("📚 Sample Anti-Pattern Demonstrations")
print("=" * 50)

sample_queries = SampleQueries.get_all_queries()

# Show first 3 examples
for key, query_info in list(sample_queries.items())[:3]:
    print(f"\n🔍 {query_info['name']}")
    print(f"Description: {query_info['description']}")
    print(
        f"Expected Anti-patterns: {', '.join(query_info['expected_antipatterns'])}")
    print(f"Query: {query_info['query'][:100]}...")

    # Test with API if available
    if config.get('deploy_api') and config.get('api_service_url'):
        try:
            analyzer = AntiPatternAnalyzer(config)
            result = analyzer.call_api(query_info['query'])

            if 'error' not in result:
                antipatterns = result.get('antipatterns', [])
                detected = [ap.get('name') for ap in antipatterns]
                print(
                    f"✅ Detected: {', '.join(detected) if detected else 'None'}")
            else:
                print(f"❌ API Error: {result['error']}")
        except Exception as e:
            print(f"❌ Test Error: {e}")

    print("-" * 30)

## Step 11: Deployment Summary

Complete overview of all deployed resources and usage instructions:

In [ ]:
# Display comprehensive deployment summary
print("🎉 Deployment Summary")
print("=" * 60)

# Build summary HTML
summary_sections = []

# Project info
summary_sections.append(f"""
<div style="background-color: #f0f8ff; padding: 20px; border-radius: 10px; margin: 10px 0;">
    <h3>✅ Deployment Completed Successfully!</h3>

    <h4>📋 Project Configuration:</h4>
    <ul>
        <li><strong>Project ID:</strong> {config.get('project_id')}</li>
        <li><strong>Region:</strong> {config.get('region')}</li>
        <li><strong>BigQuery Dataset:</strong> {config.get('bq_dataset')}</li>
        <li><strong>Artifact Registry:</strong> {config.get('artifact_registry')}</li>
    </ul>
""")

# Deployed resources
if config.get('deploy_batch'):
    summary_sections.append(f"""
    <h4>🔄 Cloud Run Job (Batch Processing):</h4>
    <ul>
        <li><strong>Job Name:</strong> {config.get('batch_job_name')}</li>
        <li><strong>Output Table:</strong> {config.get('batch_output_table')}</li>
        <li><strong>Usage:</strong> Processes INFORMATION_SCHEMA queries from last 24 hours</li>
        <li><strong>Execute:</strong> <code>gcloud run jobs execute {config.get('batch_job_name')} --region={config.get('region')} --project={config.get('project_id')}</code></li>
    </ul>
    """)

if config.get('deploy_api'):
    summary_sections.append(f"""
    <h4>🌐 Cloud Run Service (REST API):</h4>
    <ul>
        <li><strong>Service Name:</strong> {config.get('api_service_name')}</li>
        <li><strong>Service URL:</strong> <a href="{config.get('api_service_url')}" target="_blank">{config.get('api_service_url')}</a></li>
        <li><strong>Endpoint:</strong> POST {config.get('api_service_url')}/analyze</li>
        <li><strong>Usage:</strong> Send SQL queries via HTTP for real-time analysis</li>
    </ul>
    """)

if config.get('deploy_udf'):
    summary_sections.append(f"""
    <h4>🔧 BigQuery Remote UDF:</h4>
    <ul>
        <li><strong>Function Name:</strong> {config.get('udf_name')}</li>
        <li><strong>Connection:</strong> {config.get('bq_connection')}</li>
        <li><strong>Usage:</strong> Call directly in SQL queries</li>
        <li><strong>Example:</strong> <code>SELECT {config.get('udf_name')}('SELECT * FROM table') as antipatterns</code></li>
    </ul>
    """)

# Usage examples
summary_sections.append(f"""
    <h4>🚀 Quick Start Examples:</h4>
    <ol>
        <li><strong>Batch Analysis:</strong> Schedule the Cloud Run Job to run daily for automated monitoring</li>
        <li><strong>API Integration:</strong> Integrate the REST API into your CI/CD pipeline</li>
        <li><strong>SQL Analysis:</strong> Use the UDF directly in BigQuery for ad-hoc analysis</li>
    </ol>

    <h4>📚 Next Steps:</h4>
    <ul>
        <li>Explore the sample queries in the SampleQueries class</li>
        <li>Set up Cloud Scheduler for automated batch processing</li>
        <li>Create dashboards using the batch results table</li>
        <li>Integrate the API into your development workflow</li>
    </ul>
</div>
""")

# Combine and display
summary_html = ''.join(summary_sections)
display(HTML(summary_html))

# Save final configuration
config.set('deployment_status', 'completed')
config.set('deployment_completed_at', datetime.now().isoformat())

print("\n💾 Configuration saved to config.json")
print("\n🎯 All deployments completed successfully!")

## Troubleshooting & Cleanup

### Common Issues:

1. **Authentication Errors**
   ```bash
   gcloud auth login
   gcloud auth application-default login
   ```

2. **API Not Enabled**
   ```bash
   gcloud services enable cloudbuild.googleapis.com run.googleapis.com \
       artifactregistry.googleapis.com bigquery.googleapis.com \
       bigqueryconnection.googleapis.com --project=YOUR_PROJECT_ID
   ```

3. **Build Failures**
   - Check that you're in the correct directory
   - Verify cloudbuild files exist in demo/ directory
   - Check Cloud Build logs in the console

4. **Permission Issues**
   - Ensure your account has necessary IAM roles
   - Check that billing is enabled on the project

### Clean Up Resources (if needed):

```bash
# Delete Cloud Run Job
gcloud run jobs delete JOB_NAME --region=REGION --project=PROJECT_ID

# Delete Cloud Run Service
gcloud run services delete SERVICE_NAME --region=REGION --project=PROJECT_ID

# Delete Artifact Registry repository
gcloud artifacts repositories delete REPO_NAME --location=REGION --project=PROJECT_ID

# Delete BigQuery dataset
bq rm -r -d PROJECT_ID:DATASET_NAME
```

### Monitoring & Maintenance:

- **Cloud Run Logs**: Check logs in Cloud Console for debugging
- **BigQuery Jobs**: Monitor query performance and costs
- **Scheduled Execution**: Set up Cloud Scheduler for regular batch processing
- **Alerting**: Configure alerts for failed executions or high costs

---

**🎉 Congratulations! You've successfully deployed the BigQuery Anti-Pattern Recognition tool in all three scenarios!**